today we learn a concept that every production RAG systems uses

#### Create a large document

In [1]:
document = """
Artificial Intelligence is a field of computer science.
Machine Learning is a subset of Artificial Intelligence.
Deep Learning uses neural networks.
Python is widely used in AI development.
Natural Language Processing helps computers understand human language.
Computer Vision helps machines understand images.
"""

#### Manual chunking

In [2]:
chunks = [
    "Artificial Intelligence is a field of computer science.",
    "Machine Learning is a subset of Artificial Intelligence.",
    "Deep Learning uses neural networks.",
    "Python is widely used in AI development.",
    "Natural Language Processing helps computers understand human language.",
    "Computer Vision helps machines understand images."
]

print(len(chunks))

6


#### Create Embeddings

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(chunks)

print(chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(6, 384)


#### Create FAISS Index

In [4]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))

#### Query

In [5]:
query = "What is NLP?"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding),
    k=2
)

#### Retrieve Chunks

In [7]:
for idx in indices[0]:
    print(chunks[idx])

Natural Language Processing helps computers understand human language.
Machine Learning is a subset of Artificial Intelligence.


#### Try with multiple queries

In [8]:
queries = [
    "What is NLP?",
    "What is AI?",
    "What is Python?",
    "What is Deep Learning?"
]

for query in queries:
    print(f"\nQuery: {query}")

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k=3
    )

    for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
        print(f"{rank}. Distance: {distance:.4f}")
        print(f"   {chunks[idx]}")


Query: What is NLP?
1. Distance: 1.1262
   Natural Language Processing helps computers understand human language.
2. Distance: 1.4614
   Machine Learning is a subset of Artificial Intelligence.
3. Distance: 1.5010
   Deep Learning uses neural networks.

Query: What is AI?
1. Distance: 0.6189
   Artificial Intelligence is a field of computer science.
2. Distance: 0.8032
   Python is widely used in AI development.
3. Distance: 0.8536
   Machine Learning is a subset of Artificial Intelligence.

Query: What is Python?
1. Distance: 0.5990
   Python is widely used in AI development.
2. Distance: 1.2652
   Artificial Intelligence is a field of computer science.
3. Distance: 1.3855
   Natural Language Processing helps computers understand human language.

Query: What is Deep Learning?
1. Distance: 0.5449
   Deep Learning uses neural networks.
2. Distance: 0.9194
   Machine Learning is a subset of Artificial Intelligence.
3. Distance: 1.1269
   Artificial Intelligence is a field of computer sc